# WOE/IV 계산 및 Drive 저장

Google Colab에서 실행합니다. `ml/` 하위의 모든 `ml-01` 이상 실험 폴더를 자동 탐색해 처리합니다. (ml-00은 테스트용 프로토타입으로 제외)

**입력 파일**
- `MyDrive/Graph_AML/ml/{ml_folder}/{run_id}/{prefix}_feature_columns.json` — 계산 대상 컬럼
- `MyDrive/Graph_AML/ml/{ml_folder}/{mlNN}_feature_catalog.csv` — 피처 설명 (미등록 피처 체크용)
- `MyDrive/Graph_AML/data/ml/*.parquet` — 데이터 (1개, 모든 실험 공유)

**저장 파일** (`MyDrive/Graph_AML/data/ml/woe_iv/{ml-NN}/{run_id}/{model_run_id}/` 하위)
- `iv_summary.json` — feature별 IV 요약
- `bin_table.json` — 전체 bin 통계 (WOE 차트용)
- `woe_meta.json` — 계산 메타데이터

**캐시** (`MyDrive/Graph_AML/data/ml/woe_iv/woe_iv_cache.json`)
- 동일 `(woe_folder/run_id/model_run_id)` 조합은 재계산 생략

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## CONFIG — 필요시 이 섹션만 수정하세요

In [ ]:
from pathlib import Path

DRIVE_BASE_DIR = Path("/content/drive/MyDrive/Graph_AML")
ML_DIR         = DRIVE_BASE_DIR / "ml"
DATA_ML_DIR    = DRIVE_BASE_DIR / "data" / "ml"
WOE_IV_DIR     = DATA_ML_DIR / "woe_iv"   # Drive 최종 저장 루트

CACHE_PATH = WOE_IV_DIR / "woe_iv_cache.json"

# 로컬 임시 경로 (FUSE 끊김 방지)
LOCAL_OUT_DIR   = Path("/content/woe_iv_out")   # 결과 파일 로컬 임시 저장
LOCAL_CACHE_PATH = Path("/content/woe_iv_cache.json")  # 캐시 로컬 사본

# True = 캐시 무시하고 전부 재계산
FORCE_RECOMPUTE = False

# 특정 폴더만 처리 (빈 리스트 = 모든 실험)
# 예: ["ml-01", "ml-02"]
ONLY_FOLDERS: list[str] = []

# 전체 데이터 사용 여부 (False = 200k 샘플, 탐색용)
FULL_RUN = True
N_BINS   = 10
N_JOBS   = -1

## Engine — 수정하지 마세요

In [ ]:
from __future__ import annotations

import datetime
import gc
import json
import re
import time
from dataclasses import dataclass
from typing import List, Optional, Set, Tuple

import numpy as np
import pandas as pd
import pyarrow.parquet as _pq
from joblib import Parallel, delayed

# ── 상수 ──────────────────────────────────────────────────────────────────

OTHER_LABEL   = "__OTHER__"
MISSING_LABEL = "__MISSING__"

BIN_COLS = [
    "feature_name", "feature_type", "bin_id", "bin_label",
    "count", "positive_count", "negative_count", "positive_rate",
    "total_positive_rate", "positive_dist", "negative_dist", "woe", "iv_bin",
    "missing_flag", "binning_method",
]

IV_COLS = [
    "feature_name", "feature_type", "binning_method", "n_bins",
    "total_count", "positive_count", "negative_count", "iv", "iv_strength",
    "missing_count", "unique_count", "note",
]

_RESULT_FILES = ["iv_summary.json", "bin_table.json", "woe_meta.json"]


@dataclass
class Config:
    sample_mode: bool = True
    sample_n: int = 200_000
    random_state: int = 42
    n_bins_numeric: int = 10
    cat_top_n: int = 50
    cat_min_count: int = 100
    smoothing_alpha: float = 0.5
    n_jobs: int = -1


# ── Binning helpers ────────────────────────────────────────────────────────

def fit_numeric_edges(s: pd.Series, n_bins: int) -> Optional[np.ndarray]:
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty or s.nunique() < 2:
        return None
    q = min(n_bins, int(s.nunique()))
    try:
        _, edges = pd.qcut(s, q=q, retbins=True, duplicates="drop")
        edges = np.array(edges, dtype=float)
    except Exception:
        uniq = np.sort(s.unique())
        if len(uniq) < 2:
            return None
        edges = np.concatenate([uniq[:1], (uniq[:-1] + uniq[1:]) / 2, uniq[-1:]])
    if len(edges) < 3:
        return None
    edges[0], edges[-1] = -np.inf, np.inf
    return edges


def apply_numeric_edges(s: pd.Series, edges: np.ndarray) -> pd.Series:
    return pd.cut(pd.to_numeric(s, errors="coerce"), bins=edges, include_lowest=True, duplicates="drop")


def fit_cat_levels(s: pd.Series, top_n: int, min_count: int) -> Set[object]:
    counts = s.astype("object").value_counts(dropna=True)
    if counts.empty:
        return set()
    keep = counts[counts >= min_count]
    return set((keep.head(top_n) if not keep.empty else counts.head(top_n)).index)


def apply_cat_levels(s: pd.Series, keep_set: Set[object]) -> pd.Series:
    so = s.astype("object")
    return so.where(so.isin(keep_set) | so.isna(), other=OTHER_LABEL)


def compute_bin_stats(bin_series: pd.Series, y: pd.Series, alpha: float,
                      total_pos: float = None, total_neg: float = None) -> pd.DataFrame:
    # numpy bincount 기반 — pandas groupby 대비 피처당 수배 빠름
    na_mask    = bin_series.isna().to_numpy()
    labels_arr = np.where(na_mask, MISSING_LABEL, bin_series.astype(str).to_numpy())
    y_arr      = y.to_numpy(dtype=np.float64)

    uniq_labels, inverse = np.unique(labels_arr, return_inverse=True)
    n_uniq     = len(uniq_labels)
    counts     = np.bincount(inverse, minlength=n_uniq).astype(np.float64)
    pos_counts = np.bincount(inverse, weights=y_arr, minlength=n_uniq)
    neg_counts = counts - pos_counts

    if total_pos is None:
        total_pos = float(pos_counts.sum())
    if total_neg is None:
        total_neg = float(neg_counts.sum())

    n_bins   = max(n_uniq, 1)
    pos_rate = np.where(counts > 0, pos_counts / counts, np.nan)
    pos_dist = (pos_counts + alpha) / (total_pos + alpha * n_bins)
    neg_dist = (neg_counts + alpha) / (total_neg + alpha * n_bins)
    woe_vals = np.log(pos_dist / neg_dist)
    iv_vals  = (pos_dist - neg_dist) * woe_vals

    return pd.DataFrame({
        "bin_label":          uniq_labels,
        "count":              counts.astype(int),
        "positive_count":     pos_counts.astype(int),
        "negative_count":     neg_counts.astype(int),
        "positive_rate":      pos_rate,
        "total_positive_rate": float(total_pos) / max(float(counts.sum()), 1.0),
        "positive_dist":      pos_dist,
        "negative_dist":      neg_dist,
        "woe":                woe_vals,
        "iv_bin":             iv_vals,
        "missing_flag":       uniq_labels == MISSING_LABEL,
    })


def iv_strength(iv: float) -> str:
    if pd.isna(iv): return "na"
    if iv < 0.02:   return "useless"
    if iv < 0.10:   return "weak"
    if iv < 0.30:   return "medium"
    if iv < 0.50:   return "strong"
    return "suspicious"


def _process_feature(df: pd.DataFrame, feature_name: str, feature_type: str, cfg: Config,
                     total_pos: float = None, total_neg: float = None):
    y             = df["label"]
    s             = df[feature_name]
    unique_count  = int(s.nunique(dropna=True))
    missing_count = int(s.isna().sum())

    if feature_type == "numeric":
        edges = fit_numeric_edges(s, cfg.n_bins_numeric)
        if edges is None:
            return None, {
                "feature_name": feature_name, "feature_type": feature_type,
                "binning_method": "quantile", "n_bins": 0,
                "total_count": len(df), "positive_count": int(y.sum()),
                "negative_count": int((y == 0).sum()),
                "iv": np.nan, "iv_strength": "na",
                "missing_count": missing_count, "unique_count": unique_count,
                "note": "skipped: not enough unique numeric values",
            }
        binned     = apply_numeric_edges(s, edges)
        bin_id_map = {str(cat): i for i, cat in enumerate(binned.cat.categories)}
        bin_id_map[MISSING_LABEL] = -1
        method = f"quantile_n={len(edges) - 1}"
    else:
        keep_set   = fit_cat_levels(s, cfg.cat_top_n, cfg.cat_min_count)
        binned     = apply_cat_levels(s, keep_set)
        ordered    = sorted([str(x) for x in keep_set]) + [OTHER_LABEL, MISSING_LABEL]
        bin_id_map = {label: i for i, label in enumerate(ordered)}
        method = f"cat_top_n={cfg.cat_top_n}_min_count={cfg.cat_min_count}"

    tbl = compute_bin_stats(binned, y, cfg.smoothing_alpha, total_pos=total_pos, total_neg=total_neg)
    tbl["feature_name"]   = feature_name
    tbl["feature_type"]   = feature_type
    tbl["binning_method"] = method
    # lambda → .map(dict) 벡터화
    tbl["bin_id"] = tbl["bin_label"].map(bin_id_map).fillna(-99).astype(int)
    tbl = tbl[BIN_COLS]

    iv_total = float(tbl["iv_bin"].sum())
    notes = []
    if int((tbl["positive_count"] == 0).sum()):
        notes.append(f"zero_pos_bins={int((tbl['positive_count']==0).sum())}")
    if int((tbl["negative_count"] == 0).sum()):
        notes.append(f"zero_neg_bins={int((tbl['negative_count']==0).sum())}")
    if feature_type == "categorical" and (tbl["bin_label"] == OTHER_LABEL).any():
        notes.append(f"OTHER_count={int(tbl.loc[tbl['bin_label'].eq(OTHER_LABEL), 'count'].sum())}")

    iv_row = {
        "feature_name":   feature_name,
        "feature_type":   feature_type,
        "binning_method": method,
        "n_bins":         int(len(tbl)),
        "total_count":    int(tbl["count"].sum()),
        "positive_count": int(tbl["positive_count"].sum()),
        "negative_count": int(tbl["negative_count"].sum()),
        "iv":             iv_total,
        "iv_strength":    iv_strength(iv_total),
        "missing_count":  int(tbl.loc[tbl["missing_flag"], "count"].sum()),
        "unique_count":   unique_count,
        "note":           "; ".join(notes),
    }
    return tbl, iv_row


def classify_explicit_features(
    df: pd.DataFrame, feature_columns: list[str]
) -> Tuple[List[str], List[str]]:
    # 존재하는 컬럼 필터링
    valid = [c for c in feature_columns if c in df.columns]
    for c in set(feature_columns) - set(valid):
        print(f"  WARNING: '{c}' parquet에 없음, 건너뜀")
    if not valid:
        return [], []

    # nunique를 한 번에 계산 (개별 호출 대비 I/O 절감)
    nuniques = df[valid].nunique()
    numeric     = [c for c in valid if pd.api.types.is_numeric_dtype(df[c]) and nuniques[c] > 5]
    categorical = [c for c in valid if not (pd.api.types.is_numeric_dtype(df[c]) and nuniques[c] > 5)]
    return numeric, categorical


def compute_woe_iv_explicit(
    df: pd.DataFrame, feature_columns: list[str], cfg: Config
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    numeric_features, categorical_features = classify_explicit_features(df, feature_columns)
    all_features = (
        [(f, "numeric")     for f in numeric_features] +
        [(f, "categorical") for f in categorical_features]
    )
    y         = df["label"]
    total_pos = float(y.sum())
    total_neg = float(len(y) - total_pos)
    results = Parallel(n_jobs=cfg.n_jobs, prefer="threads")(
        delayed(_process_feature)(df, feat, ftype, cfg, total_pos, total_neg)
        for feat, ftype in all_features
    )
    bin_parts = [tbl for tbl, _ in results if tbl is not None]
    iv_rows   = [row for _, row in results]
    del results
    bin_df = pd.concat(bin_parts, ignore_index=True) if bin_parts else pd.DataFrame(columns=BIN_COLS)
    del bin_parts
    iv_df  = (
        pd.DataFrame(iv_rows)[IV_COLS]
        .sort_values("iv", ascending=False)
        .reset_index(drop=True)
    )
    return bin_df, iv_df


# ── 캐시 ──────────────────────────────────────────────────────────────────

def load_cache(cache_path) -> dict:
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))
    return {}


def save_cache(cache: dict, cache_path) -> None:
    cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")


def needs_recompute(cache: dict, cache_key: str, save_dir) -> bool:
    """cache_key = 'woe_folder/run_id/model_run_id'"""
    if cache_key not in cache:
        return True
    missing = [f for f in _RESULT_FILES if not (save_dir / f).exists()]
    if missing:
        print(f"  ⚠ 캐시 있지만 파일 누락: {missing} → 재계산")
        return True
    return False


# ── 경로 헬퍼 ─────────────────────────────────────────────────────────────

def _woe_iv_folder_name(ml_folder: str) -> str:
    m = re.match(r"(ml-\d+)", ml_folder)
    return m.group(1) if m else ml_folder


def _catalog_source_path(ml_folder: str, ml_dir) -> "Path":
    label = _woe_iv_folder_name(ml_folder).replace("-", "")
    return ml_dir / ml_folder / f"{label}_feature_catalog.csv"


def _artifact_prefix(rep: dict) -> str:
    return f"{rep['experiment_id']}__{rep['run_id']}__{rep['model_run_id']}"


# ── 데이터 로더 ───────────────────────────────────────────────────────────

def find_ml_parquet(data_ml_dir) -> Optional["Path"]:
    parquets = sorted(data_ml_dir.glob("*.parquet"))
    if not parquets:
        return None
    if len(parquets) > 1:
        print(f"  WARNING: {len(parquets)}개 parquet 발견 → 첫 번째 사용: {parquets[0].name}")
    return parquets[0]


def load_feature_columns(ml_dir, ml_folder: str, prefix: str, run_id: str) -> list[str]:
    path = ml_dir / ml_folder / run_id / f"{prefix}_feature_columns.json"
    if not path.exists():
        raise FileNotFoundError(f"feature_columns.json 없음: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    return data.get("feature_columns", data) if isinstance(data, dict) else list(data)


print("Engine 로드 완료")

## 실행

In [ ]:
import shutil

# ── 초기화 ────────────────────────────────────────────────────────────────
WOE_IV_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# 캐시: 로컬 사본 우선 로드 (세션 재개 시), 없으면 Drive에서 로드
cache = load_cache(LOCAL_CACHE_PATH) if LOCAL_CACHE_PATH.exists() else load_cache(CACHE_PATH)

cfg = Config(
    sample_mode=not FULL_RUN,
    n_bins_numeric=N_BINS,
    n_jobs=N_JOBS,
)

# ── parquet 확인 및 변경 감지 ─────────────────────────────────────────────
parquet_path = find_ml_parquet(DATA_ML_DIR)
if parquet_path is None:
    raise FileNotFoundError(f"parquet 없음: {DATA_ML_DIR}")

_pq_stat     = parquet_path.stat()
_pq_meta     = _pq.read_metadata(parquet_path)
_pq_snapshot = {
    "name":     parquet_path.name,
    "size":     _pq_stat.st_size,
    "num_rows": _pq_meta.num_rows,
    "mtime":    _pq_stat.st_mtime,
}
_cached_pq  = cache.get("_parquet", {})
_pq_changed = any(_pq_snapshot[k] != _cached_pq.get(k) for k in _pq_snapshot)

if _pq_changed and not FORCE_RECOMPUTE:
    changed_keys = [k for k in _pq_snapshot if _pq_snapshot[k] != _cached_pq.get(k)]
    print(f"⚠ parquet 변경 감지 ({', '.join(changed_keys)}) → 전 실험 재계산")
    for k in changed_keys:
        print(f"  {k}: {_cached_pq.get(k, '없음')} → {_pq_snapshot[k]}")

# ── 전체 실험 폴더 스캔 ────────────────────────────────────────────────────
ml_exp_dirs = sorted([
    d for d in ML_DIR.iterdir()
    if d.is_dir() and re.match(r"ml-\d+", d.name) and d.name != "ml-00"
])
print(f"발견된 ml 실험 폴더: {[d.name for d in ml_exp_dirs]}")

# ── 재계산 대상 수집 ──────────────────────────────────────────────────────
# work_items: (rep, feature_cols, catalog_df, registered, cache_key, save_dir, local_save_dir)
work_items = []
_catalog_cache: dict = {}  # ml_folder → (catalog_df, registered) — Drive 읽기 횟수 절감

for ml_exp_dir in ml_exp_dirs:
    ml_folder  = ml_exp_dir.name
    woe_folder = _woe_iv_folder_name(ml_folder)

    if ONLY_FOLDERS and woe_folder not in ONLY_FOLDERS:
        continue

    # catalog는 ml_folder당 한 번만 Drive에서 읽기
    if ml_folder not in _catalog_cache:
        catalog_source = _catalog_source_path(ml_folder, ML_DIR)
        if catalog_source.exists():
            _cdf = pd.read_csv(catalog_source, encoding="utf-8-sig")
            _cdf.columns = _cdf.columns.str.strip()
            if "피쳐명" in _cdf.columns:
                _cdf = _cdf.rename(columns={"피쳐명": "피처명"})
            _reg = set(_cdf["피처명"].tolist()) if "피처명" in _cdf.columns else set()
        else:
            _cdf, _reg = None, set()
        _catalog_cache[ml_folder] = (_cdf, _reg)
    catalog_df, registered = _catalog_cache[ml_folder]

    run_dirs = sorted([d for d in ml_exp_dir.iterdir() if d.is_dir()])
    if not run_dirs:
        print(f"[{woe_folder}] run 폴더 없음 — 건너뜀")
        continue

    for run_dir in run_dirs:
        run_id  = run_dir.name
        metrics = sorted([
            re.match(r"(.+)_metrics_val\.json$", f.name).group(1)
            for f in run_dir.iterdir()
            if re.match(r"(.+)_metrics_val\.json$", f.name)
        ])
        if not metrics:
            print(f"[{woe_folder}/{run_id}] metrics_val.json 없음 — 건너뜀")
            continue

        for prefix in metrics:
            parts        = prefix.split("__", 2)
            model_run_id = parts[2] if len(parts) >= 3 else prefix
            rep = {
                "ml_folder":     ml_folder,
                "run_id":        run_id,
                "experiment_id": parts[0] if len(parts) >= 1 else "",
                "model_run_id":  model_run_id,
            }

            # Drive 저장 경로
            save_dir       = WOE_IV_DIR     / woe_folder / run_id / model_run_id
            # 로컬 임시 저장 경로
            local_save_dir = LOCAL_OUT_DIR  / woe_folder / run_id / model_run_id
            save_dir.mkdir(parents=True, exist_ok=True)
            local_save_dir.mkdir(parents=True, exist_ok=True)

            cache_key = f"{woe_folder}/{run_id}/{model_run_id}"

            skip = (
                not FORCE_RECOMPUTE
                and not _pq_changed
                and not needs_recompute(cache, cache_key, save_dir)
            )
            if skip:
                print(f"[{woe_folder}/{run_id}/{model_run_id}] ✓ 캐시 유효 — 건너뜀")
                continue

            try:
                feature_cols = load_feature_columns(ML_DIR, ml_folder, prefix, run_id)
            except FileNotFoundError:
                print(f"[{woe_folder}/{run_id}/{model_run_id}] feature_columns.json 없음 — 건너뜀")
                continue

            work_items.append((rep, feature_cols, catalog_df, registered,
                               cache_key, save_dir, local_save_dir))

print(f"\n재계산 대상: {len(work_items)}개 실험")


# ── parquet 로드 ──────────────────────────────────────────────────────────
if not work_items:
    print("모든 실험이 캐시 유효 — parquet 로드 생략")
    df_shared = None
else:
    # Drive FUSE 연결 끊김 방지: 로컬 디스크로 먼저 복사 후 읽기
    local_pq = Path("/content") / parquet_path.name
    if not local_pq.exists():
        size_gb = _pq_stat.st_size / 1e9
        print(f"parquet Drive → 로컬 복사 중... ({size_gb:.2f} GB)")
        shutil.copy2(parquet_path, local_pq)
        print("복사 완료")
    else:
        print(f"로컬 캐시 사용: {local_pq.name}")

    _pq_schema = set(_pq.read_schema(local_pq).names)
    all_feat   = {c for _, fcols, _, _, _, _, _ in work_items for c in fcols}
    _load_cols = [c for c in list(all_feat) + ["label"] if c in _pq_schema]

    print(f"parquet 로드: {local_pq.name}  (컬럼 {len(_load_cols)}개 / 전체 {len(_pq_schema)}개)")
    df_shared = pd.read_parquet(local_pq, columns=_load_cols)
    print(f"  shape: {df_shared.shape}  |  positive rate: {df_shared['label'].mean():.5f}")
    if not FULL_RUN:
        df_shared = df_shared.sample(cfg.sample_n, random_state=cfg.random_state).reset_index(drop=True)
        print(f"  (샘플링 후: {len(df_shared):,}행)")


# ── 실험별 처리 루프 ──────────────────────────────────────────────────────
for rep, feature_cols, catalog_df, registered, cache_key, save_dir, local_save_dir in work_items:
    ml_folder    = rep["ml_folder"]
    run_id       = rep["run_id"]
    model_run_id = rep["model_run_id"]
    prefix       = _artifact_prefix(rep)
    woe_folder   = _woe_iv_folder_name(ml_folder)

    print(f"\n{'='*60}")
    print(f"[{woe_folder}/{run_id}/{model_run_id}] WOE/IV 계산 시작")
    t0 = time.time()

    print(f"  feature_columns: {len(feature_cols)}개")

    if catalog_df is not None:
        unregistered = [f for f in feature_cols if f not in registered]
        if unregistered:
            n = len(unregistered)
            print(f"  ⚠ 카탈로그 미등록 {n}개: {unregistered[:5]}{'...' if n > 5 else ''}")
        else:
            print(f"  카탈로그: {len(catalog_df)}행 ({_catalog_source_path(ml_folder, ML_DIR).name})")
    else:
        unregistered = []
        print(f"  WARNING: 카탈로그 없음 ({_catalog_source_path(ml_folder, ML_DIR)})")

    print("  WOE/IV 계산 중...")
    avail_cols = [c for c in feature_cols if c in df_shared.columns]
    df_run = df_shared[avail_cols + ["label"]]
    bin_df, iv_df = compute_woe_iv_explicit(df_run, feature_cols, cfg)
    del df_run

    elapsed = time.time() - t0
    print(f"  완료 ({elapsed:.1f}초)  |  분석 feature: {iv_df['iv'].notna().sum()}개  |  bin 행: {len(bin_df):,}")

    woe_meta = {
        "computed_at":         datetime.datetime.now().isoformat(),
        "woe_folder":          woe_folder,
        "ml_folder":           ml_folder,
        "experiment_id":       rep["experiment_id"],
        "run_id":              run_id,
        "model_run_id":        model_run_id,
        "prefix":              prefix,
        "parquet_file":        parquet_path.name,
        "full_run":            FULL_RUN,
        "n_bins_numeric":      N_BINS,
        "n_rows":              int(len(df_shared) if not FULL_RUN else _pq_meta.num_rows),
        "positive_rate":       float(df_shared["label"].mean()),
        "n_features_analyzed": int(iv_df["iv"].notna().sum()),
        "unregistered_count":  len(unregistered),
        "elapsed_seconds":     round(elapsed, 1),
    }

    # ── 로컬에 먼저 저장 ───────────────────────────────────────────────────
    iv_df.to_json(local_save_dir / "iv_summary.json",  orient="records", force_ascii=False, indent=2)
    bin_df.to_json(local_save_dir / "bin_table.json",   orient="records", force_ascii=False)
    with open(local_save_dir / "woe_meta.json", "w", encoding="utf-8") as f:
        json.dump(woe_meta, f, indent=2, ensure_ascii=False)

    del bin_df, iv_df
    gc.collect()

    # ── Drive로 복사 ───────────────────────────────────────────────────────
    for fname in _RESULT_FILES:
        shutil.copy2(local_save_dir / fname, save_dir / fname)

    # ── 캐시: 로컬 저장 후 Drive 복사 ─────────────────────────────────────
    cache["_parquet"] = _pq_snapshot
    cache[cache_key]  = {"computed_at": woe_meta["computed_at"]}
    save_cache(cache, LOCAL_CACHE_PATH)          # 로컬 빠른 쓰기
    shutil.copy2(LOCAL_CACHE_PATH, CACHE_PATH)   # Drive 복사

    print(f"  저장 완료: {save_dir}")
    for fname in _RESULT_FILES:
        fpath = save_dir / fname
        if fpath.exists():
            print(f"  {fname:<35} {fpath.stat().st_size / 1024:>7.1f} KB")

print("\n모든 실험 처리 완료")
